In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# !git clone https://github.com/Messi-Q/GNNSCVulDetector.git
%%capture
!unzip /content/GNNSCVulDetector.zip
!mv /content/content/GNNSCVulDetector /content/GNNSCVulDetector

In [ ]:
%%capture
!pip install docopt

In [ ]:
%cd /content/GNNSCVulDetector

/content/GNNSCVulDetector


In [ ]:
# @title AutoExtractGraph.py
import os
import re
import time
import numpy as np

# map user-defined variables to symbolic names(var)
var_list = ['balances[msg.sender]', 'participated[msg.sender]', 'playerPendingWithdrawals[msg.sender]',
            'nonces[msgSender]', 'balances[beneficiary]', 'transactions[transactionId]', 'tokens[token][msg.sender]',
            'totalDeposited[token]', 'tokens[0][msg.sender]', 'accountBalances[msg.sender]', 'accountBalances[_to]',
            'creditedPoints[msg.sender]', 'balances[from]', 'withdrawalCount[from]', 'balances[recipient]',
            'investors[_to]', 'Bal[msg.sender]', 'Accounts[msg.sender]', 'Holders[_addr]', 'balances[_pd]',
            'ExtractDepositTime[msg.sender]', 'Bids[msg.sender]', 'participated[msg.sender]', 'deposited[_participant]',
            'Transactions[TransHash]', 'm_txs[_h]', 'balances[investor]', 'this.balance', 'proposals[_proposalID]',
            'accountBalances[accountAddress]', 'Chargers[id]', 'latestSeriesForUser[msg.sender]',
            'balanceOf[_addressToRefund]', 'tokenManage[token_]', 'milestones[_idMilestone]', 'payments[msg.sender]',
            'rewardsForA[recipient]', 'userBalance[msg.sender]', 'credit[msg.sender]', 'credit[to]', 'round_[_rd]',
            'userPendingWithdrawals[msg.sender]', '[msg.sender]', '[from]', '[to]', '[_to]', "msg.sender"]

# function limit type
function_limit = ['private', 'onlyOwner', 'internal', 'onlyGovernor', 'onlyCommittee', 'onlyAdmin', 'onlyPlayers',
                  'onlyManager', 'onlyHuman', 'only_owner', 'onlyCongressMembers', 'preventReentry', 'onlyMembers',
                  'onlyProxyOwner', 'ownerExists', 'noReentrancy', 'notExecuted', 'noReentrancy', 'noEther',
                  'notConfirmed']

# Boolean condition expression:
var_op_bool = ['!', '~', '**', '*', '!=', '<', '>', '<=', '>=', '==', '<<', '>>', '||', '&&']

# Assignment expressions
var_op_assign = ['|=', '=', '^=', '&=', '<<=', '>>=', '+=', '-=', '*=', '/=', '%=', '++', '--']


# split all functions of contracts
def split_function(filepath):
    function_list = []
    f = open(filepath, 'r', encoding='utf-8')
    lines = f.readlines()
    f.close()
    flag = -1
    flag1 = 0

    for line in lines:
        text = line.strip()
        if len(text) > 0 and text != "\n":
            if text.split()[0] == "function" and len(function_list) > 0:
                flag1 = 0
        if flag1 == 0:
            if len(text) > 0 and text != "\n":
                if text.split()[0] == "function" or text.split()[0] == "function()":
                    function_list.append([text])
                    flag += 1
                elif len(function_list) > 0 and ("function" in function_list[flag][0]):
                    if text.split()[0] != "modifier" and text.split()[0] != "event":
                        function_list[flag].append(text)
                    else:
                        flag1 += 1
                        continue
        else:
            continue

    return function_list


# generate a potential fallback node
#def generate_potential_fallback_node(node_feature, edge_feature):
#    node_feature.append(["F", "F", "NoLimit", ["S"], 0, "MSG"])
#    edge_feature.append(["S", "F", "S", 0, "FW"])
#    edge_feature.append(["F", "W0", "F", 1, "FW"])
#    return node_feature, edge_feature


# Position the call.value to generate the graph
def generate_graph(filepath):
    allFunctionList = split_function(filepath)  # Store all functions
    callValueList = []  # Store all W functions that call call.value
    cFunctionList = []  # Store a single C function that calls a W function
    CFunctionLists = []  # Store all C functions that call W function
    withdrawNameList = []  # Store the W function name that calls call.value
    otherFunctionList = []  # Store functions other than W functions
    node_list = []  # Store all the points
    edge_list = []  # Store edge and edge nips_features
    node_feature_list = []  # Store nodes feature
    params = []  # Store the parameters of the W functions
    param = []
    key_count = 0  # Number of core nodes S and W
    c_count = 0  # Number of core nodes C

    # ======================================================================
    # ---------------------------  Handle nodes  ----------------------------
    # ======================================================================

    # Store functions other than W functions
    for i in range(len(allFunctionList)):
        flag = 0
        for j in range(len(allFunctionList[i])):
            text = allFunctionList[i][j]
            if '.call.value' in text:
                flag += 1
        if flag == 0:
            otherFunctionList.append(allFunctionList[i])

    # Traverse all functions, find the call.value keyword, store the S and W nodes
    for i in range(len(allFunctionList)):
        for j in range(len(allFunctionList[i])):
            text = allFunctionList[i][j]
            if '.call.value' in text:
                node_list.append("S")
                node_list.append("W" + str(key_count))
                callValueList.append([allFunctionList[i], "S", "W" + str(key_count)])

                # get the function name and params
                ss = allFunctionList[i][0]
                pp = re.compile(r'[(](.*?)[)]', re.S)
                result = re.findall(pp, ss)
                result_params = result[0].split(",")

                for n in range(len(result_params)):
                    param.append(result_params[n].strip().split(" ")[-1])

                params.append([param, "S", "W" + str(key_count)])

                # Handling W function access restrictions, which can be used for access restriction properties
                # default that there are C nodes
                limit_count = 0
                for k in range(len(function_limit)):
                    if function_limit[k] in callValueList[key_count][0][0]:
                        limit_count += 1
                        if "address" in text:
                            node_feature_list.append(
                                ["S", "S", "LimitedAC", ["W" + str(key_count)],
                                 2, "INNADD"])
                            node_feature_list.append(
                                ["W" + str(key_count), "W" + str(key_count), "LimitedAC", [],
                                 1, "NULL"])
                            break
                        elif "msg.sender" in text:
                            node_feature_list.append(
                                ["S", "S", "LimitedAC", ["W" + str(key_count)],
                                 2, "MSG"])
                            node_feature_list.append(
                                ["W" + str(key_count), "W" + str(key_count), "LimitedAC", [],
                                 1, "NULL"])
                            break
                        else:
                            param_count = 0
                            for pa in param:
                                if pa in text and pa != "":
                                    param_count += 1
                                    node_feature_list.append(
                                        ["S", "S", "LimitedAC",
                                         ["W" + str(key_count)],
                                         2, "MSG"])
                                    node_feature_list.append(
                                        ["W" + str(key_count), "W" + str(key_count), "LimitedAC", [],
                                         1, "NULL"])
                                    break
                            if param_count == 0:
                                node_feature_list.append(
                                    ["S", "S", "LimitedAC", ["W" + str(key_count)],
                                     2, "INNADD"])
                                node_feature_list.append(
                                    ["W" + str(key_count), "W" + str(key_count), "LimitedAC", [],
                                     1, "NULL"])
                            break
                if limit_count == 0:
                    if "address" in text:
                        node_feature_list.append(
                            ["S", "S", "NoLimit", ["W" + str(key_count)],
                             2, "INNADD"])
                        node_feature_list.append(
                            ["W" + str(key_count), "W" + str(key_count), "NoLimit", [],
                             1, "NULL"])
                    elif "msg.sender" in text:
                        node_feature_list.append(
                            ["S", "S", "NoLimit", ["W" + str(key_count)],
                             2, "MSG"])
                        node_feature_list.append(
                            ["W" + str(key_count), "W" + str(key_count), "NoLimit", [],
                             1, "NULL"])
                    else:
                        param_count = 0
                        for pa in param:
                            if pa in text and pa != "":
                                param_count += 1
                                node_feature_list.append(
                                    ["S", "S", "NoLimit", ["W" + str(key_count)],
                                     2, "MSG"])
                                node_feature_list.append(
                                    ["W" + str(key_count), "W" + str(key_count), "NoLimit", [],
                                     1, "NULL"])
                                break
                        if param_count == 0:
                            node_feature_list.append(
                                ["S", "S", "NoLimit", ["W" + str(key_count)],
                                 2, "INNADD"])
                            node_feature_list.append(
                                ["W" + str(key_count), "W" + str(key_count), "NoLimit", [],
                                 1, "NULL"])

                # For example: function transfer(address _to, uint _value, bytes _data, string _custom_fallback)
                # get function name (transfer)
                tmp = re.compile(r'\b([_A-Za-z]\w*)\b(?:(?=\s*\w+\()|(?!\s*\w+))')
                result_withdraw = tmp.findall(allFunctionList[i][0])
                withdrawNameTmp = result_withdraw[1]
                if withdrawNameTmp == "payable":
                    withdrawName = withdrawNameTmp
                else:
                    withdrawName = withdrawNameTmp + "("
                withdrawNameList.append(["W" + str(key_count), withdrawName])

                key_count += 1

    if key_count == 0:
        print("Currently, there is no key word call.value")
        node_feature_list.append(["S", "S", "NoLimit", ["NULL"], 0, "NULL"])
        node_feature_list.append(["W0", "W0", "NoLimit", ["NULL"], 0, "NULL"])
        node_feature_list.append(["C0", "C0", "NoLimit", ["NULL"], 0, "NULL"])
    else:
        # Traverse all functions and find the C function nodes that calls the W function
        # (determine the function call by matching the number of arguments)
        for k in range(len(withdrawNameList)):
            w_key = withdrawNameList[k][0]
            w_name = withdrawNameList[k][1]
            for i in range(len(otherFunctionList)):
                if len(otherFunctionList[i]) > 2:
                    for j in range(1, len(otherFunctionList[i])):
                        text = otherFunctionList[i][j]
                        if w_name in text:
                            p = re.compile(r'[(](.*?)[)]', re.S)
                            result = re.findall(p, text)
                            result_params = result[0].split(",")

                            if result_params[0] != "" and len(result_params) == len(params[k][0]):
                                cFunctionList += otherFunctionList[i]
                                CFunctionLists.append(
                                    [w_key, w_name, "C" + str(c_count), otherFunctionList[i]])
                                node_list.append("C" + str(c_count))

                                for n in range(len(node_feature_list)):
                                    if w_key in node_feature_list[n][0]:
                                        node_feature_list[n][3].append("C" + str(c_count))

                                # Handling C function access restrictions
                                limit_count = 0
                                for m in range(len(function_limit)):
                                    if function_limit[m] in cFunctionList[0]:
                                        limit_count += 1
                                        node_feature_list.append(
                                            ["C" + str(c_count), "C" + str(c_count), "LimitedAC", ["NULL"], 0, "NULL"])
                                        break
                                if limit_count == 0:
                                    node_feature_list.append(
                                        ["C" + str(c_count), "C" + str(c_count), "NoLimit", ["NULL"], 0, "NULL"])
                                c_count += 1
                                break

        if c_count == 0:
            print("There is no C node")
            node_list.append("C0")
            node_feature_list.append(["C0", "C0", "NoLimit", ["NULL"], 0, "NULL"])
            for n in range(len(node_feature_list)):
                if "W" in node_feature_list[n][0]:
                    node_feature_list[n][3] = ["NULL"]

        # ======================================================================
        # ---------------------------  Handle edge  ----------------------------
        # ======================================================================

        # (1) W->S (include: W->VAR, VAR->S, S->VAR)
        for i in range(len(callValueList)):
            flag = 0  # flag: flag = 0, before call.value; flag > 0, after call.value
            before_var_count = 0
            after_var_count = 0
            var_tmp = []
            var_name = []
            var_w_name = []
            for j in range(len(callValueList[i][0])):
                text = callValueList[i][0][j]
                if '.call.value' not in text:
                    if flag == 0:
                        # print("before call.value")
                        # handle W -> VAR
                        for k in range(len(var_list)):
                            if var_list[k] in text:
                                node_list.append("VAR" + str(before_var_count))
                                var_tmp.append("VAR" + str(before_var_count))

                                if len(var_w_name) == 0:
                                    if "assert" in text:
                                        edge_list.append(
                                            [callValueList[i][2], "VAR" + str(before_var_count), callValueList[i][2], 1,
                                             'AH'])
                                    elif "require" in text:
                                        edge_list.append(
                                            [callValueList[i][2], "VAR" + str(before_var_count), callValueList[i][2], 1,
                                             'RG'])
                                    elif j >= 1:
                                        if "if" in callValueList[i][0][j - 1]:
                                            edge_list.append(
                                                [callValueList[i][2], "VAR" + str(before_var_count),
                                                 callValueList[i][2], 1,
                                                 'GN'])
                                        elif "for" in callValueList[i][0][j - 1]:
                                            edge_list.append(
                                                [callValueList[i][2], "VAR" + str(before_var_count),
                                                 callValueList[i][2], 1,
                                                 'FOR'])
                                        elif "else" in callValueList[i][0][j - 1]:
                                            edge_list.append(
                                                [callValueList[i][2], "VAR" + str(before_var_count),
                                                 callValueList[i][2], 1,
                                                 'GB'])
                                        elif j + 1 < len(callValueList[i][0]):
                                            if "if" and "throw" in callValueList[i][0][j] or "if" in \
                                                    callValueList[i][0][j] \
                                                    and "throw" in callValueList[i][0][j + 1]:
                                                edge_list.append(
                                                    [callValueList[i][2], "VAR" + str(before_var_count),
                                                     callValueList[i][2], 1, 'IT'])
                                            elif "if" and "revert" in callValueList[i][0][j] or "if" in \
                                                    callValueList[i][0][
                                                        j] and "revert" in callValueList[i][0][j + 1]:
                                                edge_list.append(
                                                    [callValueList[i][2], "VAR" + str(before_var_count),
                                                     callValueList[i][2], 1, 'RH'])
                                            elif "if" in text:
                                                edge_list.append(
                                                    [callValueList[i][2], "VAR" + str(before_var_count),
                                                     callValueList[i][2], 1, 'IF'])
                                            else:
                                                edge_list.append(
                                                    [callValueList[i][2], "VAR" + str(before_var_count),
                                                     callValueList[i][2], 1, 'FW'])
                                        else:
                                            edge_list.append(
                                                [callValueList[i][2], "VAR" + str(before_var_count),
                                                 callValueList[i][2], 1,
                                                 'FW'])
                                    else:
                                        edge_list.append(
                                            [callValueList[i][2], "VAR" + str(before_var_count), callValueList[i][2], 1,
                                             'FW'])

                                    var_node = 0
                                    var_bool_node = 0
                                    for b in range(len(var_op_bool)):
                                        if var_op_bool[b] in text:
                                            node_feature_list.append(
                                                ["VAR" + str(before_var_count), "VAR" + str(before_var_count),
                                                 callValueList[i][2], 1, 'BOOL'])
                                            var_node += 1
                                            var_bool_node += 1
                                            break

                                    if var_bool_node == 0:
                                        for a in range(len(var_op_assign)):
                                            if var_op_assign[a] in text:
                                                node_feature_list.append(
                                                    ["VAR" + str(before_var_count), "VAR" + str(before_var_count),
                                                     callValueList[i][2], 1, 'ASSIGN'])
                                                var_node += 1
                                                break

                                    if var_node == 0:
                                        node_feature_list.append(
                                            ["VAR" + str(before_var_count), "VAR" + str(before_var_count),
                                             callValueList[i][2], 1, 'NULL'])

                                    var_w_name.append(var_list[k])
                                    var_name.append(var_list[k])
                                    before_var_count += 1
                                else:
                                    var_w_count = 0
                                    for n in range(len(var_w_name)):
                                        if var_list[k] == var_w_name[n]:
                                            var_w_count += 1
                                            var_tmp.append(var_tmp[len(var_tmp) - 1])

                                            var_node = 0
                                            var_bool_node = 0
                                            for b in range(len(var_op_bool)):
                                                if var_op_bool[b] in text:
                                                    node_feature_list.append(
                                                        [var_tmp[len(var_tmp) - 1], var_tmp[len(var_tmp) - 1],
                                                         callValueList[i][2], 1, 'BOOL'])
                                                    var_bool_node += 1
                                                    var_node += 1
                                                    break

                                            if var_bool_node == 0:
                                                for a in range(len(var_op_assign)):
                                                    if var_op_assign[a] in text:
                                                        node_feature_list.append(
                                                            [var_tmp[len(var_tmp) - 1], var_tmp[len(var_tmp) - 1],
                                                             callValueList[i][2], 1, 'ASSIGN'])
                                                        var_node += 1
                                                        break

                                            if var_node == 0:
                                                node_feature_list.append(
                                                    [var_tmp[len(var_tmp) - 1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][2], 1, 'NULL'])

                                    if var_w_count == 0:
                                        var_node = 0
                                        var_bool_node = 0
                                        var_tmp.append("VAR" + str(before_var_count))

                                        for b in range(len(var_op_bool)):
                                            if var_op_bool[b] in text:
                                                node_feature_list.append(
                                                    ["VAR" + str(before_var_count), "VAR" + str(before_var_count),
                                                     callValueList[i][2], 1, 'BOOL'])
                                                var_node += 1
                                                var_bool_node += 1
                                                break

                                        if var_bool_node == 0:
                                            for a in range(len(var_op_assign)):
                                                if var_op_assign[a] in text:
                                                    node_feature_list.append(
                                                        ["VAR" + str(before_var_count), "VAR" + str(before_var_count),
                                                         callValueList[i][2], 1, 'ASSIGN'])
                                                    var_node += 1
                                                    break

                                        if var_node == 0:
                                            node_feature_list.append(
                                                ["VAR" + str(before_var_count), "VAR" + str(before_var_count),
                                                 callValueList[i][2], 1, 'NULL'])

                    elif flag != 0:
                        # print("after call.value")
                        # handle S->VAR
                        var_count = 0
                        for k in range(len(var_list)):
                            if var_list[k] in text:
                                if before_var_count == 0:
                                    node_list.append("VAR" + str(after_var_count))
                                    var_tmp.append("VAR" + str(after_var_count))

                                    if "assert" in text:
                                        edge_list.append(
                                            [callValueList[i][1], "VAR" + str(after_var_count), callValueList[i][1], 3,
                                             'AH'])
                                    elif "require" in text:
                                        edge_list.append(
                                            [callValueList[i][1], "VAR" + str(after_var_count), callValueList[i][1], 3,
                                             'RG'])
                                    elif "return" in text:
                                        edge_list.append(
                                            [callValueList[i][1], "VAR" + str(after_var_count), callValueList[i][1], 3,
                                             'RE'])
                                    elif "if" and "throw" in text:
                                        edge_list.append(
                                            [callValueList[i][1], "VAR" + str(after_var_count), callValueList[i][1], 3,
                                             'IT'])
                                    elif "if" and "revert" in text:
                                        edge_list.append(
                                            [callValueList[i][1], "VAR" + str(after_var_count), callValueList[i][1], 3,
                                             'RH'])
                                    elif "if" in text:
                                        edge_list.append(
                                            [callValueList[i][1], "VAR" + str(after_var_count), callValueList[i][1], 3,
                                             'IF'])
                                    else:
                                        edge_list.append(
                                            [callValueList[i][1], "VAR" + str(after_var_count), callValueList[i][1], 3,
                                             'FW'])

                                    var_node = 0
                                    var_bool_node = 0
                                    for b in range(len(var_op_bool)):
                                        if var_op_bool[b] in text:
                                            node_feature_list.append(
                                                ["VAR" + str(after_var_count), "VAR" + str(after_var_count),
                                                 callValueList[i][1], 3, 'BOOL'])
                                            var_node += 1
                                            var_bool_node += 1
                                            break

                                    if var_bool_node == 0:
                                        for a in range(len(var_op_assign)):
                                            if var_op_assign[a] in text:
                                                node_feature_list.append(
                                                    ["VAR" + str(after_var_count), "VAR" + str(after_var_count),
                                                     callValueList[i][1], 3, 'ASSIGN'])
                                                var_node += 1
                                                break

                                    if var_node == 0:
                                        node_feature_list.append(
                                            ["VAR" + str(after_var_count), "VAR" + str(after_var_count),
                                             callValueList[i][1], 3, 'NULL'])

                                    # after_var_count += 1

                                elif before_var_count > 0:
                                    for n in range(len(var_name)):
                                        if var_list[k] == var_name[n]:
                                            var_count += 1
                                            if "assert" in text:
                                                edge_list.append(
                                                    [callValueList[i][1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][1], 3,
                                                     'AH'])
                                            elif "require" in text:
                                                edge_list.append(
                                                    [callValueList[i][1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][1], 3,
                                                     'RG'])
                                            elif "return" in text:
                                                edge_list.append(
                                                    [callValueList[i][1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][1], 3,
                                                     'RE'])
                                            elif "if" and "throw" in text:
                                                edge_list.append(
                                                    [callValueList[i][1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][1], 3,
                                                     'IT'])
                                            elif "if" and "revert" in text:
                                                edge_list.append(
                                                    [callValueList[i][1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][1], 3,
                                                     'RH'])
                                            elif "if" in text:
                                                edge_list.append(
                                                    [callValueList[i][1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][1], 3,
                                                     'IF'])
                                            else:
                                                edge_list.append(
                                                    [callValueList[i][1], var_tmp[len(var_tmp) - 1],
                                                     callValueList[i][1], 3,
                                                     'FW'])

                                            after_var_count += 1

                elif '.call.value' in text:
                    flag += 1

                    if len(var_tmp) > 0:
                        if "assert" in text:
                            edge_list.append(
                                [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'AH'])
                        elif "require" in text:
                            edge_list.append(
                                [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'RG'])
                        elif "return" in text:
                            edge_list.append(
                                [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'RE'])
                        elif j > 1:
                            if "if" in callValueList[i][0][j - 1]:
                                edge_list.append(
                                    [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'GN'])
                            elif "for" in callValueList[i][0][j - 1]:
                                edge_list.append(
                                    [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'FOR'])
                            elif "else" in callValueList[i][0][j - 1]:
                                edge_list.append(
                                    [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'GB'])
                            elif j + 1 < len(callValueList[i][0]):
                                if "if" and "throw" in callValueList[i][0][j] or "if" in callValueList[i][0][j] \
                                        and "throw" in callValueList[i][0][j + 1]:
                                    edge_list.append(
                                        [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'IT'])
                                elif "if" and "revert" in callValueList[i][0][j] or "if" in callValueList[i][0][j] \
                                        and "revert" in callValueList[i][0][j + 1]:
                                    edge_list.append(
                                        [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'RH'])
                                elif "if" in text:
                                    edge_list.append(
                                        [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'IF'])
                                else:
                                    edge_list.append(
                                        [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'FW'])
                            else:
                                edge_list.append(
                                    [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'FW'])
                        else:
                            edge_list.append(
                                [var_tmp[len(var_tmp) - 1], callValueList[i][1], callValueList[i][2], 2, 'FW'])

                    elif len(var_tmp) == 0:
                        if "assert" in text:
                            edge_list.append(
                                [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'AH'])
                        elif "require" in text:
                            edge_list.append(
                                [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'RG'])
                        elif "return" in text:
                            edge_list.append(
                                [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'RE'])
                        elif j > 1:
                            if "if" in callValueList[i][0][j - 1]:
                                edge_list.append(
                                    [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'GN'])
                            elif "for" in callValueList[i][0][j - 1]:
                                edge_list.append(
                                    [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'FOR'])
                            elif "else" in callValueList[i][0][j - 1]:
                                edge_list.append(
                                    [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'GB'])
                            elif j + 1 < len(callValueList[i][0]):
                                if "if" and "throw" in callValueList[i][0][j] or "if" in callValueList[i][0][j] \
                                        and "throw" in callValueList[i][0][j + 1]:
                                    edge_list.append(
                                        [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'IT'])
                                elif "if" and "revert" in callValueList[i][0][j] or "if" in callValueList[i][0][j] \
                                        and "revert" in callValueList[i][0][j + 1]:
                                    edge_list.append(
                                        [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'RH'])
                                elif "if" in text:
                                    edge_list.append(
                                        [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'IF'])
                                else:
                                    edge_list.append(
                                        [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'FW'])
                            else:
                                edge_list.append(
                                    [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'FW'])
                        else:
                            edge_list.append(
                                [callValueList[i][2], callValueList[i][1], callValueList[i][2], 1, 'FW'])

        # (2) handle C->W (include C->VAR, VAR->W)
        for i in range(len(CFunctionLists)):
            for j in range(len(CFunctionLists[i][3])):
                text = CFunctionLists[i][3][j]
                var_flag = 0
                for k in range(len(var_list)):
                    if var_list[k] in text:
                        var_flag += 1

                        var_node = 0
                        var_bool_node = 0
                        for b in range(len(var_op_bool)):
                            if var_op_bool[b] in text:
                                node_feature_list.append(
                                    ["VAR" + str(len(var_tmp)), "VAR" + str(len(var_tmp)),
                                     CFunctionLists[i][2], 1, 'BOOL'])
                                var_node += 1
                                var_bool_node += 1
                                break

                        if var_bool_node == 0:
                            for a in range(len(var_op_assign)):
                                if var_op_assign[a] in text:
                                    node_feature_list.append(
                                        ["VAR" + str(len(var_tmp)), "VAR" + str(len(var_tmp)),
                                         CFunctionLists[i][2], 1, 'ASSIGN'])
                                    var_node += 1
                                    break

                        if var_node == 0:
                            node_feature_list.append(
                                ["VAR" + str(len(var_tmp)), "VAR" + str(len(var_tmp)),
                                 CFunctionLists[i][2], 1, 'NULL'])

                        if "assert" in text:
                            edge_list.append(
                                [CFunctionLists[i][2], "VAR" + str(len(var_tmp)), CFunctionLists[i][2], 1, 'AH'])
                            edge_list.append(
                                ["VAR" + str(len(var_tmp)), CFunctionLists[i][0], CFunctionLists[i][2], 2, 'FW'])
                        elif "require" in text:
                            edge_list.append(
                                [CFunctionLists[i][2], "VAR" + str(len(var_tmp)), CFunctionLists[i][2], 1, 'RG'])
                            edge_list.append(
                                ["VAR" + str(len(var_tmp)), CFunctionLists[i][0], CFunctionLists[i][2], 2, 'FW'])
                        elif "if" and "throw" in text:
                            edge_list.append(
                                [CFunctionLists[i][2], "VAR" + str(len(var_tmp)), CFunctionLists[i][2], 1, 'IT'])
                            edge_list.append(
                                ["VAR" + str(len(var_tmp)), CFunctionLists[i][0], CFunctionLists[i][2], 2, 'FW'])
                        elif "if" and "revert" in text:
                            edge_list.append(
                                [CFunctionLists[i][2], "VAR" + str(len(var_tmp)), CFunctionLists[i][2], 1, 'RH'])
                            edge_list.append(
                                ["VAR" + str(len(var_tmp)), CFunctionLists[i][0], CFunctionLists[i][2], 2, 'FW'])
                        elif "if" in text:
                            edge_list.append(
                                [CFunctionLists[i][2], "VAR" + str(len(var_tmp)), CFunctionLists[i][2], 1, 'IF'])
                            edge_list.append(
                                ["VAR" + str(len(var_tmp)), CFunctionLists[i][0], CFunctionLists[i][2], 2, 'FW'])
                        else:
                            edge_list.append(
                                [CFunctionLists[i][2], "VAR" + str(len(var_tmp)), CFunctionLists[i][2], 1, 'FW'])
                            edge_list.append(
                                ["VAR" + str(len(var_tmp)), CFunctionLists[i][0], CFunctionLists[i][2], 2, 'FW'])
                        break

                if var_flag == 0:
                    if "assert" in text:
                        edge_list.append(
                            [CFunctionLists[i][2], CFunctionLists[i][0], CFunctionLists[i][2], 1, 'AH'])
                    elif "require" in text:
                        edge_list.append(
                            [CFunctionLists[i][2], CFunctionLists[i][0], CFunctionLists[i][2], 1, 'RG'])
                    elif "if" and "throw" in text:
                        edge_list.append(
                            [CFunctionLists[i][2], CFunctionLists[i][0], CFunctionLists[i][2], 1, 'IT'])
                    elif "if" and "revert" in text:
                        edge_list.append(
                            [CFunctionLists[i][2], CFunctionLists[i][0], CFunctionLists[i][2], 1, 'RH'])
                    elif "if" in text:
                        edge_list.append(
                            [CFunctionLists[i][2], CFunctionLists[i][0], CFunctionLists[i][2], 1, 'IF'])
                    else:
                        edge_list.append(
                            [CFunctionLists[i][2], CFunctionLists[i][0], CFunctionLists[i][2], 1, 'FW'])
                    break
                else:
                    print("The C function does not call the corresponding W function")

    # Handling some duplicate elements, the filter leaves a unique
    edge_list = list(set([tuple(t) for t in edge_list]))
    edge_list = [list(v) for v in edge_list]
    node_feature_list_new = []
    [node_feature_list_new.append(i) for i in node_feature_list if not i in node_feature_list_new]
    # node_feature_list = list(set([tuple(t) for t in node_feature_list]))
    # node_feature_list = [list(v) for v in node_feature_list]
    # node_list = list(set(node_list))

    return node_feature_list_new, edge_list


def printResult(file, node_feature, edge_feature, label):
    main_point = ['S', 'W0', 'W1', 'W2', 'W3', 'W4', 'C0', 'C1', 'C2', 'C3', 'C4']

    for i in range(len(node_feature)):
        if node_feature[i][0] in main_point:
            for j in range(0, len(node_feature[i][3]), 2):
                if j + 1 < len(node_feature[i][3]):
                    tmp = node_feature[i][3][j] + "," + node_feature[i][3][j + 1]
                elif len(node_feature[i][3]) == 1:
                    tmp = node_feature[i][3][j]

            node_feature[i][3] = tmp

    nodeOutPath = f"data/reentrancy/graph_data/node/{file[:-4]}_{label}"
    edgeOutPath = f"data/reentrancy/graph_data/edge/{file[:-4]}_{label}"

    f_node = open(nodeOutPath, 'a')
    for i in range(len(node_feature)):
        result = " ".join(np.array(node_feature[i]))
        f_node.write(result + '\n')
    f_node.close()

    f_edge = open(edgeOutPath, 'a')
    for i in range(len(edge_feature)):
        result = " ".join(np.array(edge_feature[i]))
        # print(result)
        f_edge.write(result + '\n')
    f_edge.close()
    return node_feature, edge_feature


if __name__ == "__main__":
    test_contract = "/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeDataset/ReentrancyDataset/Train/Vulnerable/0xb6aca06a3588f4ce5ce33a1a7e9152892b250ca3.sol"
    node_feature, edge_feature = generate_graph(test_contract)
    node_feature = sorted(node_feature, key=lambda x: (x[0]))
    edge_feature = sorted(edge_feature, key=lambda x: (x[2], x[3]))
    printResult(test_contract.split("/")[-1], node_feature, edge_feature, 0)
   # node_feature, edge_feature = generate_potential_fallback_node(node_feature, edge_feature)
    print("node_feature", node_feature)
    print("edge_feature", edge_feature)

    # inputFileDir = "../../data/reentrancy/source_code/"
    # dirs = os.listdir(inputFileDir)
    # start_time = time.time()
    # for file in dirs:
    #     inputFilePath = inputFileDir + file
    #     node_feature, edge_feature = generate_graph(inputFilePath)
    #     node_feature = sorted(node_feature, key=lambda x: (x[0]))
    #     edge_feature = sorted(edge_feature, key=lambda x: (x[2], x[3]))
    #     printResult(file, node_feature, edge_feature)
    #
    # end_time = time.time()
    # print(end_time - start_time)

There is no C node
node_feature [['C0', 'C0', 'NoLimit', 'NULL', 0, 'NULL'], ['S', 'S', 'NoLimit', 'W0', 2, 'INNADD'], ['VAR0', 'VAR0', 'W0', 1, 'ASSIGN'], ['W0', 'W0', 'NoLimit', 'NULL', 1, 'NULL']]
edge_feature [['W0', 'VAR0', 'W0', 1, 'FW'], ['VAR0', 'S', 'W0', 2, 'IF']]


In [ ]:
# @title Dataset Getter
import os
import shutil

def get_links(dir):
    return [os.path.join(dir, f) for f in os.listdir(dir)]


reen_train_dataset = get_links("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/Vulnerable")
non_reen_train_dataset = get_links("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Train/NonVulnerable")
reen_test_dataset = get_links("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/Vulnerable")
non_reen_test_dataset = get_links("/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable")

In [ ]:
!rm -r data/reentrancy/graph_data/node/
!rm -r data/reentrancy/graph_data/edge/
!mkdir data/reentrancy/graph_data/node/
!mkdir data/reentrancy/graph_data/edge/

In [ ]:
dirs = {
    "train": {
        1: reen_train_dataset,
        0: non_reen_train_dataset
    },
    "test": {
        1: reen_test_dataset,
        0: non_reen_test_dataset
    }
}

In [ ]:
from tqdm import tqdm

start_time = time.time()
for stype in dirs.keys():
    for label in dirs[stype].keys():
        ds = dirs[stype][label]
        i = 0
        for file_path in tqdm(ds):
            try:
                inputFilePath = file_path
                node_feature, edge_feature = generate_graph(inputFilePath)
                node_feature = sorted(node_feature, key=lambda x: (x[0]))
                edge_feature = sorted(edge_feature, key=lambda x: (x[2], x[3]))
                printResult(file_path.split("/")[-1], node_feature, edge_feature, label)
            except Exception as e:
                print(e)
                print(inputFilePath)
                print("-" * 100)
                continue


end_time = time.time()
print(end_time - start_time)

  0%|          | 1/281 [00:00<01:08,  4.09it/s]

There is no C node


  1%|▏         | 4/281 [00:01<02:48,  1.64it/s]

There is no C node


 16%|█▌        | 44/281 [00:02<00:06, 35.47it/s]

...

list index out of range
/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x5e7356d72fb1e802c4c6bab301c30b3a693d62d1.sol
----------------------------------------------------------------------------------------------------


100%|█████████▉| 284/285 [01:00<00:00,  4.71it/s]

list index out of range
/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x55b81686957ce18b1102442022701e594585228a.sol
----------------------------------------------------------------------------------------------------
There is no C node


100%|██████████| 285/285 [01:00<00:00,  4.70it/s]

list index out of range
/content/drive/MyDrive/SmartContractVulnerabilityDetection/SourceCodeSyntaxDataset/ReentrancyDataset/Test/NonVulnerable/0x3920d11e2af18af31da6328ea5d1e12c26f50c1d.sol
----------------------------------------------------------------------------------------------------
186.52832770347595


In [ ]:
# @title graph2vec.py
import os
import json
import numpy as np
from tools.reentrancy.vec2onehot import vec2onehot

"""
S, W, C nips_features: Node nips_features + Edge nips_features + Var nips_features;
Node self property + Incoming Var + Outgoing Var + Incoming Edge + Outgoing Edge
"""

dict_AC = {"NULL": 0, "LimitedAC": 1, "NoLimit": 2}

dict_NodeName = {"NULL": 0, "VAR0": 1, "VAR1": 2, "VAR2": 3, "VAR3": 4, "VAR4": 5, "VAR5": 6, "S": 7, "W0": 8,
                 "W1": 9, "W2": 10, "W3": 11, "W4": 12, "C0": 13, "C1": 14, "C2": 15, "C3": 16, "C4": 17}

dict_VarOpName = {"NULL": 0, "BOOL": 1, "ASSIGN": 2}

dict_EdgeOpName = {"NULL": 0, "FW": 1, "IF": 2, "GB": 3, "GN": 4, "WHILE": 5, "FOR": 6, "RE": 7, "AH": 8, "RG": 9,
                   "RH": 10, "IT": 11}

dict_AllOpName = {"NULL": 0, "FW": 1, "ASSIGN": 2, "BOOL": 3, "IF": 4, "GB": 5, "GN": 6, "WHILE": 7, "FOR": 8, "RE": 9,
                  "AH": 10, "RG": 11, "RH": 12, "IT": 13}

dict_NodeOpName = {"NULL": 0, "MSG": 1, "INNADD": 2}

dict_ConName = {"NULL": 0, "ARG1": 1, "ARG2": 2, "ARG3": 3, "CON1": 4, "CON2": 5, "CON3": 6, "CNS1": 7, "CNS2": 8,
                "CNS3": 9}

node_convert = {"S": 0, "W0": 1, "C0": 2, "W1": 3, "C1": 4, "W2": 5, "C2": 6, "W3": 7, "C3": 8, "W4": 9, "C4": 10,
                "VAR0": 0, "VAR1": 1, "VAR2": "VAR2", "VAR3": "VAR3", "VAR4": "VAR4", "VAR5": "VAR5"}

v2o = vec2onehot()  # create the one-bot dicts


# extract the nips_features of each node from input file #
def extract_node_features(nodeFile):
    nodeNum = 0
    node_list = []
    node_attribute_list = []

    f = open(nodeFile)
    lines = f.readlines()
    f.close()

    for line in lines:
        node = list(map(str, line.split()))
        verExist = False
        for i in range(0, len(node_list)):
            if node[1] == node_list[i]:
                verExist = True
            else:
                continue
        if verExist is False:
            node_list.append(node[1])
            nodeNum += 1
        node_attribute_list.append(node)

    return nodeNum, node_list, node_attribute_list


# elimination procedure for sub_graph Start here #
def elimination_node(node_attribute_list):
    main_point = ['S', 'W0', 'W1', 'W2', 'W3', 'W4', 'C0', 'C1', 'C2', 'C3', 'C4']
    extra_var_list = []  # extract var with low priority
    for i in range(0, len(node_attribute_list)):
        if node_attribute_list[i][1] not in main_point:
            if i + 1 < len(node_attribute_list):
                if node_attribute_list[i][1] == node_attribute_list[i + 1][1]:
                    loc1 = int(node_attribute_list[i][3])  # relative location
                    op1 = node_attribute_list[i][4]  # operation
                    loc2 = int(node_attribute_list[i + 1][3])
                    op2 = node_attribute_list[i + 1][4]
                    if loc2 - loc1 == 1:
                        op1_index = dict_VarOpName[op1]
                        op2_index = dict_VarOpName[op2]
                        # extract node attribute based on priority
                        if op1_index < op2_index:
                            extra_var_list.append(node_attribute_list.pop(i))
                        else:
                            extra_var_list.append(node_attribute_list.pop(i + 1))
    return node_attribute_list, extra_var_list


def embedding_node(node_attribute_list):
    # embedding each node after elimination #
    node_encode = []
    var_encode = []
    node_embedding = []
    var_embedding = []
    main_point = ['S', 'W0', 'W1', 'W2', 'W3', 'W4', 'C0', 'C1', 'C2', 'C3', 'C4']

    for j in range(0, len(node_attribute_list)):
        v = node_attribute_list[j][0]
        if v in main_point:
            vf0 = node_attribute_list[j][0]
            vf1 = dict_NodeName[node_attribute_list[j][1]]
            vfm1 = v2o.node2vecEmbedding(node_attribute_list[j][1])
            vf2 = dict_AC[node_attribute_list[j][2]]
            vfm2 = v2o.nodeAC2vecEmbedding(node_attribute_list[j][2])

            result = node_attribute_list[j][3].split(",")
            for call_vec in range(len(result)):
                if call_vec + 1 < len(result):
                    tmp_vf = str(dict_NodeName[result[call_vec]]) + "," + str(dict_NodeName[result[call_vec + 1]])
                    tmp_vfm = np.array(list(v2o.node2vecEmbedding(result[call_vec]))) ^ np.array(
                        list(v2o.node2vecEmbedding(result[call_vec + 1])))
                elif len(result) == 1:
                    tmp_vf = dict_NodeName[result[call_vec]]
                    tmp_vfm = v2o.node2vecEmbedding(result[call_vec])
            vf3 = tmp_vf
            vfm3 = tmp_vfm
            vf4 = int(node_attribute_list[j][4])
            vfm4 = v2o.sn2vecEmbedding(node_attribute_list[j][4])
            vf5 = dict_NodeOpName[node_attribute_list[j][5]]
            vfm5 = v2o.nodeOP2vecEmbedding(node_attribute_list[j][5])
            nodeEmbedding = vfm1.tolist() + vfm2.tolist() + vfm3.tolist() + vfm4.tolist() + vfm5.tolist()
            node_embedding.append([vf0, np.array(nodeEmbedding)])
            temp = [vf1, vf2, vf3, vf4, vf5]
            node_encode.append([vf0, temp])
        else:
            vf0 = node_attribute_list[j][0]
            vf1 = dict_NodeName[node_attribute_list[j][1]]
            vfm1 = v2o.node2vecEmbedding(node_attribute_list[j][1])
            vf2 = dict_NodeName[node_attribute_list[j][2]]
            vfm2 = v2o.node2vecEmbedding(node_attribute_list[j][2])
            vf3 = int(node_attribute_list[j][3])
            vfm3 = v2o.sn2vecEmbedding(node_attribute_list[j][3])
            vf4 = dict_VarOpName[node_attribute_list[j][4]]
            vfm4 = v2o.varOP2vecEmbedding(node_attribute_list[j][4])
            vf5 = int(dict_NodeOpName['NULL'])
            vfm5 = v2o.nodeOP2vecEmbedding('NULL')
            varEmbedding = vfm1.tolist() + vfm2.tolist() + vfm3.tolist() + vfm4.tolist() + vfm5.tolist()
            var_embedding.append([vf0, np.array(varEmbedding)])
            temp = [vf1, vf2, vf3, vf4, vf5]
            var_encode.append([vf0, temp])

    return node_encode, var_encode, node_embedding, var_embedding


def elimination_edge(edgeFile):
    # eliminate edge #
    edge_list = []  # all edge
    extra_edge_list = []  # eliminated edge

    f = open(edgeFile)
    lines = f.readlines()
    f.close()

    for line in lines:
        edge = list(map(str, line.split()))
        edge_list.append(edge)

    # The ablation of multiple edge between two nodes, taking the edge with the edge_operation priority
    for k in range(0, len(edge_list)):
        if k + 1 < len(edge_list):
            start1 = edge_list[k][0]  # start node
            end1 = edge_list[k][1]  # end node
            op1 = edge_list[k][4]
            start2 = edge_list[k + 1][0]
            end2 = edge_list[k + 1][1]
            op2 = edge_list[k + 1][4]
            if start1 == start2 and end1 == end2:
                op1_index = dict_EdgeOpName[op1]
                op2_index = dict_EdgeOpName[op2]
                # extract edge attribute based on priority
                if op1_index < op2_index:
                    extra_edge_list.append(edge_list.pop(k))
                else:
                    extra_edge_list.append(edge_list.pop(k + 1))

    return edge_list, extra_edge_list


def embedding_edge(edge_list):
    # extract & embedding the nips_features of each edge from input file #
    edge_encode = []
    edge_embedding = []

    for k in range(len(edge_list)):
        start = edge_list[k][0]  # start node
        end = edge_list[k][1]  # end node
        a, b, c = edge_list[k][2], edge_list[k][3], edge_list[k][4]  # origin info

        ef1 = dict_NodeName[a]
        ef2 = int(b)
        ef3 = dict_EdgeOpName[c]

        ef_temp = [ef1, ef2, ef3]
        edge_encode.append([start, end, ef_temp])

        efm1 = v2o.node2vecEmbedding(a)
        efm2 = v2o.sn2vecEmbedding(b)
        efm3 = v2o.edgeOP2vecEmbedding(c)

        efm_temp = efm1.tolist() + efm2.tolist() + efm3.tolist()
        edge_embedding.append([start, end, np.array(efm_temp)])

    return edge_encode, edge_embedding


def construct_vec(edge_list, node_embedding, var_embedding, edge_embedding, edge_encode):
    # Vec: Node self property + Incoming Var + Outgoing Var + Incoming Edge + Outgoing Edge
    print("Start constructing node vector...")
    var_in_node = []
    var_in = []
    var_out_node = []
    var_out = []
    edge_in_node = []
    edge_in = []
    edge_out_node = []
    edge_out = []
    node_vec = []
    F_point = ['F']
    S_point = ['S']
    W_point = ['W0', 'W1', 'W2', 'W3', 'W4']
    C_point = ['C0', 'C1', 'C2', 'C3', 'C4']
    main_point = ['S', 'W0', 'W1', 'W2', 'W3', 'W4', 'C0', 'C1', 'C2', 'C3', 'C4']
    node_embedding_dim_without_edge = 250

    if len(var_embedding) > 0:
        for k in range(len(edge_embedding)):
            if edge_list[k][0] in F_point:
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][1]):
                        var_out.append([edge_embedding[k][0], var_embedding[i][1]])
                        edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
            elif edge_list[k][1] in F_point:
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][0]):
                        var_in.append([edge_embedding[k][1], var_embedding[i][1]])
                        edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])

            if edge_list[k][0] in C_point:
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][1]):
                        var_out.append([edge_embedding[k][0], var_embedding[i][1]])
                        edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
            elif edge_list[k][1] in C_point:
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][0]):
                        var_in.append([edge_embedding[k][1], var_embedding[i][1]])
                        edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])

            elif edge_list[k][0] in W_point:
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][1]):
                        var_out.append([edge_embedding[k][0], var_embedding[i][1]])
                        edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
                        break
            elif edge_list[k][1] in W_point:
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][0]):
                        var_in.append([edge_embedding[k][1], var_embedding[i][1]])
                        edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])

            elif edge_list[k][0] in S_point:
                S_OUT = []
                S_OUT_Flag = 0
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][1]):
                        S_OUT.append(var_embedding[i][1])
                        S_OUT_Flag = 1
                if S_OUT_Flag != 1:
                    S_OUT.append(np.zeros(len(var_embedding[0][1]), dtype=int))
                var_out.append([edge_embedding[k][0], S_OUT[0]])
                edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
            elif edge_list[k][1] in S_point:
                for i in range(len(var_embedding)):
                    if str(var_embedding[i][0]) == str(edge_embedding[k][0]):
                        var_in.append([edge_embedding[k][1], var_embedding[i][1]])
                        edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])
                        break
            else:
                print("Edge from node %s to node %s:  edgeFeature: %s" % (
                    edge_embedding[k][0], edge_embedding[k][1], edge_embedding[k][2]))
    else:
        for k in range(len(edge_embedding)):
            if edge_list[k][0] in F_point:
                edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
            elif edge_list[k][1] in F_point:
                edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])

            if edge_list[k][0] in C_point:
                edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
            elif edge_list[k][1] in C_point:
                edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])

            elif edge_list[k][0] in W_point:
                edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
            elif edge_list[k][1] in W_point:
                edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])

            elif edge_list[k][0] in S_point:
                edge_out.append([edge_embedding[k][0], edge_embedding[k][2]])
            elif edge_list[k][1] in S_point:
                edge_in.append([edge_embedding[k][1], edge_embedding[k][2]])

    edge_vec_length = 44
    var_vec_length = 61

    for i in range(len(var_in)):
        var_in_node.append(var_in[i][0])
    for i in range(len(var_out)):
        var_out_node.append(var_out[i][0])
    for i in range(len(edge_in)):
        edge_in_node.append(edge_in[i][0])
    for i in range(len(edge_out)):
        edge_out_node.append(edge_out[i][0])

    for i in range(len(main_point)):
        if main_point[i] not in var_in_node:
            var_in.append([main_point[i], np.zeros(var_vec_length, dtype=int)])
        if main_point[i] not in var_out_node:
            var_out.append([main_point[i], np.zeros(var_vec_length, dtype=int)])
        if main_point[i] not in edge_out_node:
            edge_out.append([main_point[i], np.zeros(edge_vec_length, dtype=int)])
        if main_point[i] not in edge_in_node:
            edge_in.append([main_point[i], np.zeros(edge_vec_length, dtype=int)])

    varIn_dict = dict(var_in)
    varOut_dict = dict(var_out)
    edgeIn_dict = dict(edge_in)
    edgeOut_dict = dict(edge_out)

    for i in range(len(node_embedding)):
        vec = np.zeros(node_embedding_dim_without_edge, dtype=int)
        if node_embedding[i][0] in F_point:
            node_feature = node_embedding[i][1].tolist() + np.array(varIn_dict[node_embedding[i][0]]).tolist() + \
                           np.array(varOut_dict[node_embedding[i][0]]).tolist()
            vec[0:len(np.array(node_feature))] = np.array(node_feature)
            node_vec.append([node_embedding[i][0], vec])
        elif node_embedding[i][0] in S_point:
            node_feature = node_embedding[i][1].tolist() + np.array(varIn_dict[node_embedding[i][0]]).tolist() + \
                           np.array(varOut_dict[node_embedding[i][0]]).tolist()
            vec[0:len(np.array(node_feature))] = np.array(node_feature)
            node_vec.append([node_embedding[i][0], vec])
        elif node_embedding[i][0] in W_point:
            node_feature = node_embedding[i][1].tolist() + np.array(varIn_dict[node_embedding[i][0]]).tolist() + \
                           np.array(varOut_dict[node_embedding[i][0]]).tolist()
            vec[0:len(np.array(node_feature))] = np.array(node_feature)
            node_vec.append([node_embedding[i][0], vec])
        elif node_embedding[i][0] in C_point:
            node_feature = node_embedding[i][1].tolist() + np.array(varIn_dict[node_embedding[i][0]]).tolist() + \
                           np.array(varOut_dict[node_embedding[i][0]]).tolist()
            vec[0:len(np.array(node_feature))] = np.array(node_feature)
            node_vec.append([node_embedding[i][0], vec])

    for i in range(len(node_vec)):
        node_vec[i][1] = node_vec[i][1].tolist()

    print("Node Vec:")
    for i in range(len(node_vec)):
        node_vec[i][0] = node_convert[node_vec[i][0]]
        print(node_vec[i][0], node_vec[i][1])

    for i in range(len(edge_embedding)):
        edge_embedding[i][2] = edge_embedding[i][2].tolist()

    # "S" -> 0, W0 -> 1, C0 -> 2
    if len(edge_encode) == 2:
        end = edge_encode[len(edge_encode) - 2][1]
        start = edge_encode[len(edge_encode) - 1][0]
        flag = edge_encode[len(edge_encode) - 1][1]
        if end == start and ('VAR' in flag or 'MSG' in flag):
            edge_encode[len(edge_encode) - 1][1] = edge_encode[len(edge_encode) - 2][0]

    if len(edge_encode) > 2:
        end1 = edge_encode[len(edge_encode) - 1][1]
        start2 = edge_encode[len(edge_encode) - 2][0]
        if end1 == start2 and ('VAR' in end1 or 'MSG' in end1):
            edge_encode[len(edge_encode) - 1][1] = edge_encode[len(edge_encode) - 3][0]

    for i in range(len(edge_encode)):
        if i + 1 < len(edge_encode):
            start1 = edge_encode[i][0]
            end1 = edge_encode[i][1]
            start2 = edge_encode[i + 1][0]

            if end1 == start2 and ('VAR' in end1 or 'MSG' in end1):
                edge_encode[i][1] = edge_encode[i + 1][1]
                edge_encode[i + 1][0] = edge_encode[i][0]
            elif 'W' in start1 and 'VAR' in end1:
                edge_encode[i][1] = 'S'

    print("Edge Vec:")
    for i in range(len(edge_encode)):
        edge_encode[i][0] = node_convert[edge_encode[i][0]]
        edge_encode[i][1] = node_convert[edge_encode[i][1]]
        print(edge_encode[i][0], edge_encode[i][1], edge_encode[i][2])

    graph_edge = []

    for i in range(len(edge_encode)):
        graph_edge.append([edge_encode[i][0], edge_encode[i][2][2], edge_encode[i][1]])

    print(graph_edge)

    return node_vec, graph_edge


path = "data/reentrancy/graph_data/node"
data_list = []
for file in tqdm(os.listdir(path)):
    try:
        graph_dict = {}
        print(file)
        node = f"data/reentrancy/graph_data/node/{file}"
        edge = f"data/reentrancy/graph_data/edge/{file}"
        nodeNum, node_list, node_attribute_list = extract_node_features(node)
        print(node_attribute_list)
        node_attribute_list, extra_var_list = elimination_node(node_attribute_list)
        node_encode, var_encode, node_embedding, var_embedding = embedding_node(node_attribute_list)
        edge_list, extra_edge_list = elimination_edge(edge)
        edge_encode, edge_embedding = embedding_edge(edge_list)
        node_vec, graph_edge = construct_vec(edge_list, node_embedding, var_embedding, edge_embedding, edge_encode)
        node_features = [y for x, y in node_vec]
        contract_name = node.split('/')[-1].split('_')[0] + '.sol'
        graph = graph_edge
        targets = node.split('/')[-1].split('_')[1].split('.')[0]
        graph_dict["targets"] = targets
        graph_dict["graph"] = graph
        graph_dict["contract_name"] = contract_name
        graph_dict["node_features"] = node_features
        data_list.append(graph_dict)
    except Exception as e:
        print(e)
        pass

 44%|████▍     | 226/516 [00:00<00:00, 1148.14it/s]

0xadb797366e36697dc9742a8c820d3dda931668d2_0
[['C0', 'C0', 'NoLimit', 'NULL', '0', 'NULL'], ['C1', 'C1', 'NoLimit', 'NULL', '0', 'NULL'], ['C2', 'C2', 'NoLimit', 'NULL', '0', 'NULL'], ['S', 'S', 'NoLimit', 'W0', '2', 'MSG'], ['W0', 'W0', 'NoLimit', 'C0,C1', '1', 'NULL'], ['C0', 'C0', 'NoLimit', 'NULL', '0', 'NULL'], ['C1', 'C1', 'NoLimit', 'NULL', '0', 'NULL'], ['C2', 'C2', 'NoLimit', 'NULL', '0', 'NULL'], ['S', 'S', 'NoLimit', 'W0', '2', 'MSG'], ['W0', 'W0', 'NoLimit', 'C0,C1', '1', 'NULL']]
Start constructing node vector...
Node Vec:
2 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

 91%|█████████ | 467/516 [00:00<00:00, 1103.55it/s]

0xf6e7ea2e29e64f62b667d5c6e0cf65ebb5ab5b5d_0
[['C0', 'C0', 'LimitedAC', 'NULL', '0', 'NULL'], ['C1', 'C1', 'LimitedAC', 'NULL', '0', 'NULL'], ['C2', 'C2', 'LimitedAC', 'NULL', '0', 'NULL'], ['S', 'S', 'LimitedAC', 'W0', '2', 'INNADD'], ['W0', 'W0', 'LimitedAC', 'C0,C1', '1', 'NULL'], ['C0', 'C0', 'LimitedAC', 'NULL', '0', 'NULL'], ['C1', 'C1', 'LimitedAC', 'NULL', '0', 'NULL'], ['C2', 'C2', 'LimitedAC', 'NULL', '0', 'NULL'], ['S', 'S', 'LimitedAC', 'W0', '2', 'INNADD'], ['W0', 'W0', 'LimitedAC', 'C0,C1', '1', 'NULL']]
Start constructing node vector...
Node Vec:
2 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

100%|██████████| 516/516 [00:00<00:00, 1033.80it/s]

[['C0', 'C0', 'NoLimit', 'NULL', '0', 'NULL'], ['S', 'S', 'NoLimit', 'W0', '2', 'MSG'], ['VAR0', 'VAR0', 'W0', '1', 'ASSIGN'], ['VAR1', 'VAR1', 'W0', '1', 'ASSIGN'], ['W0', 'W0', 'NoLimit', 'NULL', '1', 'NULL'], ['C0', 'C0', 'NoLimit', 'NULL', '0', 'NULL'], ['S', 'S', 'NoLimit', 'W0', '2', 'MSG'], ['VAR0', 'VAR0', 'W0', '1', 'ASSIGN'], ['VAR1', 'VAR1', 'W0', '1', 'ASSIGN'], ['W0', 'W0', 'NoLimit', 'NULL', '1', 'NULL']]
Start constructing node vector...
Node Vec:
2 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [ ]:
path = "data/reentrancy/graph_data/node"
data_list = []
for file in tqdm(os.listdir(path)):
    try:
        graph_dict = {}
        print(file)
        node = f"data/reentrancy/graph_data/node/{file}"
        edge = f"data/reentrancy/graph_data/edge/{file}"
        nodeNum, node_list, node_attribute_list = extract_node_features(node)
        print(node_attribute_list)
        node_attribute_list, extra_var_list = elimination_node(node_attribute_list)
        node_encode, var_encode, node_embedding, var_embedding = embedding_node(node_attribute_list)
        edge_list, extra_edge_list = elimination_edge(edge)
        edge_encode, edge_embedding = embedding_edge(edge_list)
        node_vec, graph_edge = construct_vec(edge_list, node_embedding, var_embedding, edge_embedding, edge_encode)
        node_features = [y for x, y in node_vec]
        contract_name = node.split('/')[-1].split('_')[0] + '.sol'
        graph = graph_edge
        targets = node.split('/')[-1].split('_')[1].split('.')[0]
        print(targets)
        print("-" * 100)
        graph_dict["targets"] = targets
        graph_dict["graph"] = graph
        graph_dict["contract_name"] = contract_name
        graph_dict["node_features"] = node_features
        data_list.append(graph_dict)
    except Exception as e:
        print(e)
        pass

 35%|███▍      | 180/516 [00:00<00:00, 1799.38it/s]

0xadb797366e36697dc9742a8c820d3dda931668d2_0
[['C0', 'C0', 'NoLimit', 'NULL', '0', 'NULL'], ['C1', 'C1', 'NoLimit', 'NULL', '0', 'NULL'], ['C2', 'C2', 'NoLimit', 'NULL', '0', 'NULL'], ['S', 'S', 'NoLimit', 'W0', '2', 'MSG'], ['W0', 'W0', 'NoLimit', 'C0,C1', '1', 'NULL'], ['C0', 'C0', 'NoLimit', 'NULL', '0', 'NULL'], ['C1', 'C1', 'NoLimit', 'NULL', '0', 'NULL'], ['C2', 'C2', 'NoLimit', 'NULL', '0', 'NULL'], ['S', 'S', 'NoLimit', 'W0', '2', 'MSG'], ['W0', 'W0', 'NoLimit', 'C0,C1', '1', 'NULL']]
Start constructing node vector...
Node Vec:
2 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

 93%|█████████▎| 482/516 [00:00<00:00, 985.62it/s]

Node Vec:
2 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
0 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

100%|██████████| 516/516 [00:00<00:00, 1001.58it/s]

Node Vec:
2 [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
0 [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [ ]:
# slit data list to train and valid json
from sklearn.model_selection import train_test_split
import json
import os
from tqdm import tqdm

train_data, valid_data = train_test_split(data_list, test_size=0.2, random_state=42)
with open('train_data/reentrancy/train.json', 'w') as f:
    json.dump(train_data, f)
with open('train_data/reentrancy/valid.json', 'w') as f:
    json.dump(valid_data, f)

In [ ]:
import json

def validate_data_file(path):
    with open(path) as f:
        data = json.load(f)
    temp_data = {}
    for idx, item in enumerate(data):
        try:
            [float(t[0]) for t in item['targets']]
            temp_data[idx] = item
        except ValueError:
            print(f"Bad entry at index {idx}: {item['targets']}")
    with open(path, 'w') as f:
        json.dump(list(temp_data.values()), f)

validate_data_file("train_data/reentrancy/train.json")
validate_data_file("train_data/reentrancy/valid.json")

Bad entry at index 161: simple


In [ ]:
!python3 GNNSCModel.py --random_seed 9930 --thresholds 0.45

2025-03-08 17:19:19.098446: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1741454359.159921    3686 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741454359.179141    3686 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-08 17:19:19.254986: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2.18.0
Run 2025-03-08-17-19-24_3686 starting with following parameters:
{"num_epochs": 250, "patience": 200, "learning_rate":

In [ ]:
import tensorflow as tf
import keras
print(tf.__version__)
print(keras.__version__)

2.17.0
3.4.1


In [ ]:
!zip -r /content/GNNSCVulDetector.zip /content/GNNSCVulDetector

Kết quả truyền trực tuyến bị cắt bớt đến 5000 dòng cuối.
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/0xa52e014b3f5cc48287c2d483a3e026c32cc76e6d_1 (deflated 39%)
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/0x74fd51a98a4a1ecbef8cc43be801cce630e260bd_0 (deflated 53%)
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/27398_1 (deflated 33%)
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/0x30f5310f99fd72180e21ea8fbdcfc3b02795007f_0 (deflated 53%)
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/0x416993d2384d9b82687f34f7fea29f6fb2c6c56d_1 (deflated 46%)
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/37676_1 (deflated 30%)
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/0xb291eb985f3a994a0f8fd84bba15538c9079005b_0 (deflated 53%)
  adding: content/GNNSCVulDetector/data/reentrancy/graph_data/node/0xf26f1cd9f6e7e28136d5300b9190cfbb6fd3bf6f_0 (deflated 53%)
 

In [ ]:
while True:
  pass

KeyboardInterrupt: 

In [ ]:
cnt = 0
for i in range(len(data_list)):
    tmp = [v for e in data_list[i]['graph'] for v in [e[0], e[2]]]
    if tmp != []:
        print(data_list[i]['graph'])
        print(tmp)
        print("-" * 100)
        cnt+=1

print(cnt)

[[1, 1, 0], [1, 2, 0]]
[1, 0, 1, 0]
----------------------------------------------------------------------------------------------------
[[2, 2, 1], [1, 1, 0], [2, 2, 1], [1, 1, 0], [2, 2, 1], [1, 1, 0]]
[2, 1, 1, 0, 2, 1, 1, 0, 2, 1, 1, 0]
----------------------------------------------------------------------------------------------------
[[1, 2, 0], [3, 2, 0]]
[1, 0, 3, 0]
----------------------------------------------------------------------------------------------------
[[1, 1, 0], [3, 7, 0], [5, 1, 0]]
[1, 0, 3, 0, 5, 0]
----------------------------------------------------------------------------------------------------
[[2, 1, 1], [4, 1, 1], [6, 1, 1], [1, 7, 0]]
[2, 1, 4, 1, 6, 1, 1, 0]
----------------------------------------------------------------------------------------------------
[[2, 1, 1], [1, 2, 0], [2, 1, 1], [1, 2, 0], [2, 1, 1], [1, 2, 0]]
[2, 1, 1, 0, 2, 1, 1, 0, 2, 1, 1, 0]
--------------------------------------------------------------------------------------------